# Cosmos DB Parallel Container Migration Validation (Fabric)

This notebook orchestrates **parallel validation** of multiple Cosmos DB container migrations by reading a CSV configuration file and calling `CosmosDBLiveContainerMigrationValidation` for each container pair.

**How it works:**
| Step | Description |
|------|-------------|
| 1 | Read CSV with container migration definitions |
| 2 | Configure parallelism |
| 3 | Build DAG and execute parallel validation notebook runs via `notebookutils.notebook.runMultiple()` |
| 4 | Collect and display results |

In [6]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "Cosmos_Migration"
    }
}

StatementMeta(, c8cff035-7370-4f42-bdbe-a40c3bb85899, -1, Finished, Available, Finished, True)

## Step 1: Read Migration Configuration CSV

Reads the same `cosmosDBLiveMigrationList.csv` used by the parallel migration notebook.

In [ ]:
# Read the migration configuration CSV from Lakehouse Files
csv_path = "Files/cosmosDBLiveMigrationList.csv"

migration_df = spark.read.option("header", True).csv(csv_path)

# Display available migrations to validate
print(f"Found {migration_df.count()} container migration(s) to validate:")
migration_df.show(truncate=False)

# Collect as list of dicts for building DAG activities
migration_list = [row.asDict() for row in migration_df.collect()]

## Step 2: Configure Parallelism

Set the maximum number of validations to run in parallel.

> **Recommendation:** Start with 4 and adjust based on cluster capacity.

In [ ]:
# Maximum number of validations to run in parallel
num_notebooks_in_parallel = 4

# Name of the child notebook to call for each container validation
notebook_to_run = "CosmosDBLiveContainerMigrationValidation"

print(f"Will run up to {num_notebooks_in_parallel} validations in parallel")
print(f"Child notebook: {notebook_to_run}")
print(f"Total containers to validate: {len(migration_list)}")

## Step 3: Build DAG & Execute Parallel Validations

Builds a DAG of parallel activities from the CSV rows and executes them using `notebookutils.notebook.runMultiple()`. Each activity calls `CosmosDBLiveContainerMigrationValidation` with the source and target parameters for that container pair.

In [ ]:
# Build DAG activities from CSV migration list
activities = []
for i, row in enumerate(migration_list):
    activity = {
        "name": f"Validate_{i}_{row['cosmosSourceDatabaseName']}_{row['cosmosSourceContainerName']}",
        "path": notebook_to_run,
        "timeoutPerCellInSeconds": 600,
        "retry": 1,
        "retryIntervalInSeconds": 30,
        "args": {
            "cosmosSourceEndpoint": row["cosmosSourceEndpoint"],
            "cosmosSourceMasterKey": row["cosmosSourceMasterKey"],
            "cosmosTargetEndpoint": row["cosmosTargetEndpoint"],
            "cosmosTargetMasterKey": row["cosmosTargetMasterKey"],
            "cosmosRegion": row["cosmosRegion"],
            "cosmosSourceDatabaseName": row["cosmosSourceDatabaseName"],
            "cosmosSourceContainerName": row["cosmosSourceContainerName"],
            "cosmosTargetDatabaseName": row["cosmosTargetDatabaseName"],
            "cosmosTargetContainerName": row["cosmosTargetContainerName"],
        }
    }
    activities.append(activity)

# Build the DAG
dag = {
    "activities": activities,
    "timeoutInSeconds": 86400,
    "concurrency": num_notebooks_in_parallel
}

# Validate DAG
print(f"DAG built with {len(activities)} activities:")
for act in activities:
    print(f"  - {act['name']}")

is_valid = notebookutils.notebook.validateDAG(dag)
print(f"\nDAG validation: {'PASSED' if is_valid else 'FAILED'}")

## Step 4: Execute Parallel Validation

This cell launches all container validations in parallel. It will block until all notebooks complete or timeout.

In [ ]:
# Execute all container validations in parallel
print(f"Starting parallel validation of {len(activities)} containers (max {num_notebooks_in_parallel} concurrent)...")
print("=" * 70)

results = notebookutils.notebook.runMultiple(dag)

# Display results
print("\n" + "=" * 70)
print("VALIDATION RESULTS:")
print("=" * 70)

success_count = 0
error_count = 0
for activity_name, result in results.items():
    if result.get("exception"):
        print(f"  FAILED  - {activity_name}: {result['exception']}")
        error_count += 1
    else:
        print(f"  SUCCESS - {activity_name}: {result.get('exitVal', 'completed')}")
        success_count += 1

print(f"\nSummary: {success_count} succeeded, {error_count} failed out of {len(results)} total")